In [ ]:
import os


In [ ]:
os.environ["CUDA_VISIBLE_DEVICES"] = "6"

In [ ]:
dir_path = "/scratch/yag1/ego4d_data/v2/clips"

mp4_files = [f for f in os.listdir(dir_path) if f.endswith(".mp4")]

print(f"Number of mp4 files: {len(mp4_files)}")

In [ ]:
# !pip install mediapy

In [ ]:
# @title Load model
import jax
import jax.numpy as jnp
from videoprism import models as vp
import numpy as np
import mediapy
import matplotlib.pyplot as plt
import cv2
from decord import VideoReader, cpu

In [ ]:
def read_video(filename: str) -> tuple[VideoReader, float]:
    """Reads a video file and returns a VideoReader object and its FPS."""
    # 1. Open video (Lazy initialization; does not read frames into memory yet)

    vr = VideoReader(filename, ctx=cpu(0))
    fps = vr.get_avg_fps()

    return vr, fps


In [ ]:
# def preprocess_video(
#     video: np.ndarray, target_num_frames: int, target_frame_size: tuple[int, int], start_time: float, end_time: float
# ):
#   """Preprocesses a video by sampling frames, resizing, and normalizing pixel values."""
  
#   video_fps = video.metadata.fps
#   start_frame = int(start_time * video_fps)
#   end_frame = int(end_time * video_fps)
  
#   frames = video[start_frame:end_frame]

#   # Sample to target number of frames.
#   frame_indices = np.linspace(
#       0, len(frames), num=target_num_frames, endpoint=False, dtype=np.int32
#   )
#   frames = np.array([frames[i] for i in frame_indices])

#   # Resize to target size.
#   original_height, original_width = frames.shape[-3:-1]
  
#   target_height, target_width = target_frame_size
#   assert (
#       original_height * target_width == original_width * target_height
#   ), 'Currently does not support aspect ratio mismatch.'
#   frames = mediapy.resize_video(frames, shape=target_frame_size)

#   # Normalize pixel values to [0.0, 1.0].
#   frames = mediapy.to_float01(frames)

#   return frames


In [ ]:
# !pip install decord

In [ ]:
def preprocess_video_decord(
    vr: VideoReader, fps: float, target_num_frames: int, target_frame_size: tuple[int, int], start_time: float, end_time: float
) -> np.ndarray:
    
    # 1. Calculate frame range bounds
    start_frame = min(int(start_time * fps), len(vr) - 1)
    end_frame = min(int(end_time * fps), len(vr) - 1)
    
    # 2. Linearly sample global frame indices directly
    frame_indices = np.linspace(
        start_frame, end_frame, num=target_num_frames, endpoint=False, dtype=np.int32
    )
    
    # 3. Decord fetches ONLY the targeted frames (massive speedup)
    frames = vr.get_batch(frame_indices).asnumpy() # Shape: (num_frames, H, W, C)
    
    num_frames, orig_h, orig_w, channels = frames.shape
    target_h, target_w = target_frame_size

    # Calculate the scaling factor required to fit the frame inside the target dimensions
    # Taking the min() automatically decides whether to scale by width or height
    scale = min(target_w / orig_w, target_h / orig_h)
    new_w = int(orig_w * scale)
    new_h = int(orig_h * scale)
    
    # Calculate the padding required to center the frame
    # (Using floor division // to get integer pixel coordinates)
    pad_top = (target_h - new_h) // 2
    pad_left = (target_w - new_w) // 2
    
    # Initialize a blank canvas array filled with zeros (black) of the target size
    resized_frames = np.zeros(
        (num_frames, target_h, target_w, channels), dtype=frames.dtype
    )
    
    for i in range(num_frames):
        # 5. Resize the frame while maintaining aspect ratio
        resized_frame = cv2.resize(frames[i], (new_w, new_h), interpolation=cv2.INTER_LINEAR)
        
        # 6. Place the resized frame onto the center of the canvas
        resized_frames[i, pad_top:pad_top+new_h, pad_left:pad_left+new_w] = resized_frame
    
    # Normalize pixel values to [0.0, 1.0].
    resized_frames = mediapy.to_float01(resized_frames)
    
    # print(f"Preprocessed video shape: {resized_frames.shape}")
    # print(f"Preprocessed video dtype: {resized_frames.dtype}")
    # print(f"Preprocessed video min and max pixel values: {resized_frames.min()}, {resized_frames.max()}")
    # plt.imshow(resized_frames[0])  # Display the first frame to verify preprocessing
    # plt.axis('off')
    # plt.show()
    
    return resized_frames

In [ ]:
# # 5. Fast batch resize using OpenCV (C++ vectorized backend)
    # target_height, target_width = target_frame_size
    # resized_frames = np.empty((target_num_frames, target_height, target_width, 3), dtype=np.uint8)
    
    # print(f"Original video segment shape: {frames.shape}")
    
    # for i, frame in enumerate(frames):
    #     # Note: cv2.resize expects (width, height) format
    #     resized_frames[i] = cv2.resize(frame, (target_width, target_height))
        
    # # 6. Fast vector normalization
    # return resized_frames.astype(np.float32) / 255.0    

In [ ]:
print(jax.devices())

In [ ]:
MODEL_NAME = 'videoprism_lvt_public_v1_base'  # @param ['videoprism_lvt_public_v1_base', 'videoprism_lvt_public_v1_large'] {allow-input: false}
USE_BFLOAT16 = False  # @param { type: "boolean" }
NUM_FRAMES = 16
FRAME_SIZE = 288

fprop_dtype = jnp.bfloat16 if USE_BFLOAT16 else None
flax_model = vp.get_model(MODEL_NAME, fprop_dtype=fprop_dtype)
loaded_state = vp.load_pretrained_weights(MODEL_NAME)
text_tokenizer = vp.load_text_tokenizer('c4_en')


@jax.jit
def forward_fn(inputs, text_token_ids, text_paddings, train=False):
  return flax_model.apply(
      loaded_state,
      inputs,
      text_token_ids,
      text_paddings,
      train=train,
  )

In [ ]:
import json
from tqdm import tqdm
import torch

In [ ]:
# Load all segment JSONs once
segments_by_duration = {}
for i in range(1, 11):
    with open(f"{i}_seconds_segments.json") as f:
        segments_by_duration[i] = json.load(f)

for mp4_file in tqdm(mp4_files):
    mp4_name = os.path.splitext(mp4_file)[0]
    os.makedirs(f"{dir_path}/{mp4_name}/video_embeddings", exist_ok=True)

    mp4_path = os.path.join(dir_path, mp4_file)
    video_reader, video_fps = read_video(mp4_path)

    for i in range(1, 11):
        segments = segments_by_duration[i][f"{dir_path}/{mp4_name}.mp4"]

        for idx, segment in enumerate(segments):
            start_time = segment[0]
            end_time = segment[1]
            
            frames = preprocess_video_decord(video_reader, video_fps, NUM_FRAMES, (FRAME_SIZE, FRAME_SIZE), start_time, end_time)
            frames = jnp.asarray(frames[None, ...])  # Add batch dimension.
            
            if USE_BFLOAT16:
                frames = frames.astype(jnp.bfloat16)
            
            seg_id = f"{i}_seconds_segment_{idx}"
            
            video_embeddings, _, _ = forward_fn(frames, None, None)
            jax_arr = jax.device_get(video_embeddings[0])
            torch_tensor = torch.tensor(np.asarray(jax_arr))
            
            torch.save(torch_tensor, f"{dir_path}/{mp4_name}/video_embeddings/{seg_id}.pt")
    
    print(f"Finished saving embeddings for {mp4_file}")